# 02 — Execution Quality Analysis

This notebook walks through the public execution-quality workflow using anonymized sample data.

The goal is to show how apparent signal edge is filtered through execution frictions such as acceptance, fill status, spread, timing, and settlement outcomes. The analysis is demonstration-only and should not be read as complete historical performance or as a live trading result.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path("..") / "data" / "sample"
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
candidates = pd.read_csv(DATA_DIR / "candidates_sample.csv")
executions = pd.read_csv(DATA_DIR / "executions_sample.csv")
rejections = pd.read_csv(DATA_DIR / "rejections_sample.csv")
settlements = pd.read_csv(DATA_DIR / "settlements_sample.csv")

for df in [candidates, executions, rejections]:
    if "recorded_at" in df.columns:
        df["recorded_at"] = pd.to_datetime(df["recorded_at"], utc=True, errors="coerce")

settlements["market_start_utc"] = pd.to_datetime(settlements["market_start_utc"], utc=True, errors="coerce")
settlements["market_end_utc"] = pd.to_datetime(settlements["market_end_utc"], utc=True, errors="coerce")

## Public sample execution funnel

The funnel below is row-based. Because the public sample is downsampled and field-filtered, it is a diagnostic view rather than a complete production audit trail.

In [ ]:
funnel = pd.DataFrame(
    [
        {"stage": "Candidate signals", "rows": len(candidates)},
        {"stage": "Execution attempts", "rows": len(executions)},
        {"stage": "Orders sent", "rows": int(executions["order_sent"].fillna(False).astype(bool).sum())},
        {"stage": "Orders accepted", "rows": int(executions["order_accepted"].fillna(False).astype(bool).sum())},
        {"stage": "Filled", "rows": int(executions["filled"].fillna(False).astype(bool).sum())},
        {"stage": "Settlements", "rows": len(settlements)},
    ]
)
funnel

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(funnel["stage"], funnel["rows"])
plt.title("Public Sample Execution Funnel")
plt.ylabel("Rows")
plt.xticks(rotation=25, ha="right")
plt.show()

## Status and rejection diagnostics

The status breakdown separates successful fills, failed attempts, and blocked attempts. Rejection categories are simplified to avoid exposing private operational details.

In [ ]:
status_breakdown = (
    executions["status"]
    .fillna("unknown")
    .value_counts()
    .rename_axis("status")
    .reset_index(name="rows")
)
status_breakdown["share"] = status_breakdown["rows"] / status_breakdown["rows"].sum()
status_breakdown

In [ ]:
rejection_breakdown = (
    rejections["rejection_reason_category"]
    .fillna("unknown")
    .value_counts()
    .rename_axis("rejection_reason_category")
    .reset_index(name="rows")
)
rejection_breakdown["share"] = rejection_breakdown["rows"] / rejection_breakdown["rows"].sum()
rejection_breakdown

## Fill rate by time bucket

Short-horizon prediction markets are sensitive to time-to-expiry. This section checks whether the public sample has materially different fill behavior across timing buckets.

In [ ]:
by_time = (
    executions.assign(filled_flag=executions["filled"].fillna(False).astype(bool))
    .groupby("time_bucket", dropna=False)
    .agg(
        rows=("status", "size"),
        fill_rate=("filled_flag", "mean"),
        avg_signal_edge=("signal_edge", "mean"),
        avg_spread=("signal_spread", "mean"),
        avg_latency_ms=("latency_ms", "mean"),
    )
    .reset_index()
    .sort_values("time_bucket", na_position="last")
)
by_time

In [ ]:
plot_data = by_time.dropna(subset=["time_bucket"])
plt.figure(figsize=(8, 4.5))
plt.bar(plot_data["time_bucket"], plot_data["fill_rate"])
plt.title("Fill Rate by Time Bucket")
plt.ylabel("Fill rate")
plt.xlabel("Elapsed-time bucket")
plt.xticks(rotation=30, ha="right")
plt.show()

## Fill rate by signal-edge bucket

A key question is whether higher apparent signal edge translates into better executable outcomes. In real markets, higher apparent edge may coincide with worse liquidity, lower fill probability, or noisier late-stage markets.

In [ ]:
def edge_bucket(edge: float) -> str:
    if pd.isna(edge):
        return "missing"
    if edge < 0.25:
        return "<0.25"
    if edge < 0.35:
        return "0.25-0.35"
    if edge < 0.50:
        return "0.35-0.50"
    return ">=0.50"

executions = executions.assign(
    filled_flag=executions["filled"].fillna(False).astype(bool),
    edge_bucket=executions["signal_edge"].apply(edge_bucket),
)

edge_summary = (
    executions.groupby("edge_bucket", dropna=False)
    .agg(
        rows=("status", "size"),
        fill_rate=("filled_flag", "mean"),
        avg_signal_edge=("signal_edge", "mean"),
        avg_spread=("signal_spread", "mean"),
        avg_latency_ms=("latency_ms", "mean"),
    )
    .reset_index()
)
edge_summary

In [ ]:
order = ["<0.25", "0.25-0.35", "0.35-0.50", ">=0.50", "missing"]
plot_data = edge_summary.set_index("edge_bucket").reindex([label for label in order if label in edge_summary["edge_bucket"].values]).reset_index()
plt.figure(figsize=(7, 4.5))
plt.bar(plot_data["edge_bucket"], plot_data["fill_rate"])
plt.title("Fill Rate by Signal-Edge Bucket")
plt.ylabel("Fill rate")
plt.xlabel("Signal-edge bucket")
plt.show()

## Spread distribution

Spread is one of the most direct frictions between a theoretical edge and an executable edge. This chart uses the anonymized `signal_spread` field from execution attempts.

In [ ]:
spread = executions["signal_spread"].dropna()
spread.describe().to_frame(name="signal_spread")

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.hist(spread, bins=20)
plt.title("Signal Spread Distribution")
plt.xlabel("Signal spread")
plt.ylabel("Execution attempts")
plt.show()

## ML and fill-probability decision diagnostics

These diagnostics summarize scalar public-sample fields exported from the private ledger. They are not a production ML model artifact and should not be interpreted as proof of predictive edge.


In [ ]:
ml_summary = pd.DataFrame(
    [
        {
            "metric": "rows_with_ml_predicted_ev",
            "value": executions["ml_predicted_ev"].notna().sum() if "ml_predicted_ev" in executions else 0,
        },
        {
            "metric": "ml_pass_rate",
            "value": executions["ml_passed"].astype(str).str.lower().isin(["true", "1", "yes"]).mean()
            if "ml_passed" in executions
            else None,
        },
        {
            "metric": "avg_ml_predicted_ev",
            "value": pd.to_numeric(executions.get("ml_predicted_ev"), errors="coerce").mean()
            if "ml_predicted_ev" in executions
            else None,
        },
        {
            "metric": "rows_with_fill_probability",
            "value": executions["fill_probability"].notna().sum() if "fill_probability" in executions else 0,
        },
        {
            "metric": "fill_probability_pass_rate",
            "value": executions["fill_prob_passed"].astype(str).str.lower().isin(["true", "1", "yes"]).mean()
            if "fill_prob_passed" in executions
            else None,
        },
        {
            "metric": "avg_fill_probability",
            "value": pd.to_numeric(executions.get("fill_probability"), errors="coerce").mean()
            if "fill_probability" in executions
            else None,
        },
    ]
)
ml_summary


In [ ]:
if "ml_reason" in executions:
    display(executions["ml_reason"].fillna("none").value_counts().rename_axis("ml_reason").reset_index(name="rows"))
if "fill_prob_reason" in executions:
    display(
        executions["fill_prob_reason"]
        .fillna("none")
        .value_counts()
        .rename_axis("fill_prob_reason")
        .reset_index(name="rows")
    )


## Settlement and normalized PnL summary

Settlement rows use normalized PnL. This supports public diagnostics without publishing private notional size or raw ledger records.

In [ ]:
pnl = settlements["net_pnl_normalized"].dropna()
pnl_summary = pd.DataFrame(
    [
        {"metric": "rows", "value": len(pnl)},
        {"metric": "average_normalized_pnl", "value": pnl.mean()},
        {"metric": "median_normalized_pnl", "value": pnl.median()},
        {"metric": "minimum_normalized_pnl", "value": pnl.min()},
        {"metric": "maximum_normalized_pnl", "value": pnl.max()},
        {"metric": "positive_pnl_rate", "value": (pnl > 0).mean()},
    ]
)
pnl_summary

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.hist(pnl, bins=30)
plt.title("Normalized Settlement PnL Distribution")
plt.xlabel("Normalized net PnL")
plt.ylabel("Markets")
plt.show()

## Interpretation

This notebook demonstrates the execution-quality analysis path:

1. Start from candidate signals and execution attempts.
2. Separate accepted, filled, failed, and blocked rows.
3. Compare fill behavior across timing, edge, and spread buckets.
4. Review settlement-level normalized PnL as a public-safe diagnostic.

The public sample is useful for methodology review and reproducibility. It is not a complete live-performance dataset, and it should not be used to infer profitability or production execution readiness.